In [10]:
import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from prompt_toolkit.styles import Priority

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

In [16]:
from typing import Optional
from pydantic import BaseModel, Field

class Person(BaseModel):
    """人物信息"""
    name:str = Field(description="姓名")
    # age:Optional[int] = Field(description="年龄")
    age:int = Field(description="年龄")
    occupation:str = Field(description="职业")

model_output = llm_deepseek.with_structured_output(Person)

res = model_output.invoke("张三是一名工程师")
print(res)

name='张三' age=30 occupation='工程师'


# 枚举

In [17]:
from typing import Optional
from pydantic import BaseModel, Field
from enum import Enum
class Priority(str,Enum):
    LOW="低"
    MIDDLE="中"
    HIGH="高"

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

In [18]:
# 测试
structured_llm = llm_deepseek.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency.value}")

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency=<Priority.HIGH: '高'>

提取结果：
 客户: 王小明
 电话: 138-1234-5678
 邮箱: 未提供
 问题: 订单一直没发货
 紧急程度: 高


# 列表提取

In [20]:
from typing import List
from pydantic import BaseModel, Field

class Person(BaseModel):
    """人物信息"""
    name:str = Field(description="姓名")
    age:int = Field(description="年龄")

class PersonList(BaseModel):
    people : List[Person] = Field(description="人群")

model = llm_deepseek.with_structured_output(PersonList)

res = model.invoke("张三 30岁 , 里斯 20岁")
print(res)

people=[Person(name='张三', age=30), Person(name='里斯', age=20)]


In [22]:
from pydantic import BaseModel, Field

class Address(BaseModel):
    """地点描述"""
    city: str = Field(description="城市")
    district: str = Field(description="区域")

class Company(BaseModel):
    """公司信息"""
    name: str = Field(description="公司名称")
    address: Address = Field(description="公司所在地")

model = llm_deepseek.with_structured_output(Company)
model.invoke("介绍阿里巴巴公司")

Company(name='阿里巴巴', address=Address(city='杭州', district='余杭区'))

# 限制条件

In [26]:
from pydantic import BaseModel, Field, ValidationError


class Person(BaseModel):
    """人物信息"""
    name:str = Field(description="姓名" , min_length=2 , max_length=4)
    age:int = Field(description="年龄" , le=100)

# person = Person(name="hhh" , age=99 )
# print(person)
try:
    person = Person(name="hgghh" , age=99 )
    print(person)
except ValidationError as e:
    print("11")

11
